# Rounded and censored assay calibration

This experiment improves the current NN + V3 candidate's Gamma_GT predictions using an assay measurement model. It treats integer readings as intervals and the minimum reading as censored. These are modeling hypotheses. The new CSV retains the previous candidate's Smoking predictions.

The local combined score improves from **0.860764 to 0.861158**. Kaggle performance is unknown. This project-backed notebook displays the full new Python implementation and saved evidence.

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Code
ROOT=Path.cwd()
for script in ['calibration_precision_push.py','assemble_precision_candidate.py']:
    display(Code(filename=str(ROOT/script),language='python'))

In [ ]:
RUN_RETRAIN=False
if RUN_RETRAIN:
    import subprocess
    for script in ['calibration_precision_push.py','assemble_precision_candidate.py']:
        subprocess.run([str(ROOT/'.venv/bin/python'),script],cwd=ROOT,check=True)
OUT=ROOT/'outputs/calibration_precision_push'
display(json.loads((OUT/'conditional_review.json').read_text()))
display(json.loads((OUT/'submission_summary.json').read_text()))

## Interpretation

The 30% blend improved on three calibration split assignments. It was chosen after examining 15% and 30% on the first two assignments and held fixed for the third. The assignments reuse the same rows; they are not independent validation.

New assay parameters exclude each calibration fold's evaluation labels. However, clinical priors and baseline predictions come from cached V2 models whose stacking is not fully nested. Both the final score and grouped bootstrap intervals are conditional diagnostics. A Kaggle test is needed to measure public performance.

Spline/Ridge, clinical SVR, and Smoking group adjustments did not provide supported improvements and were omitted.

In [ ]:
summary=json.loads((OUT/'submission_summary.json').read_text())
path=ROOT/summary['submission']
assert hashlib.sha256(path.read_bytes()).hexdigest()==summary['sha256']
candidate=pd.read_csv(path,float_precision='round_trip')
before=pd.read_csv(ROOT/'outputs/final_review/submission_nn_v3_verified.csv',float_precision='round_trip')
sample=pd.read_csv(ROOT/'data/sample_submission.csv')
assert len(candidate)==6399 and candidate.ID.equals(sample.ID)
assert candidate.columns.tolist()==['ID','Smoking','Gamma_GT']
assert np.isfinite(candidate[['Smoking','Gamma_GT']]).all().all()
assert candidate.Smoking.between(0,1).all() and candidate.Gamma_GT.between(1,1000).all()
np.testing.assert_array_equal(candidate.Smoking,before.Smoking)
print('Verified candidate:',path)